# Build an observable store replenishment agent with the Agents API

A store has four cases of bottled water on the shelf and twenty in the back room. The agent first recommends a shelf refill. Later, a storm delays the inbound truck and increases expected demand, so the **same Agents API session** recommends a transfer from a nearby store.

```text
low shelf alert -> restock from back room
        |
storm delays truck -> request nearby-store transfer
```

You will see the dataset, application tools, Agents API calls, tool activity, decisions, and trace identifiers.

## Setup

Run this notebook from `examples/agents_api`. Store `OPENAI_API_KEY` in the repository's ignored `.env.local` file; do not paste it into a notebook cell. The setup cell loads that file before creating the client. The application key needs `api.agents.read`, `api.agents.write`, and `api.responses.write`.

In [ ]:
%pip install "openai>=3.13.0" "ipywidgets>=8.1.0" "python-dotenv>=1.0.0"

In [ ]:
import importlib
import json
import os
import sys
import tempfile
from pathlib import Path

import ipywidgets as widgets
from dotenv import load_dotenv
from IPython.display import Markdown, clear_output, display
from openai import APIError, OpenAI


REPO_ROOT = next((root for root in (Path.cwd(), *Path.cwd().parents)
                  if (root / "examples/agents_api/build-observable-store-replenishment/replenishment_agent.py").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open the notebook from the Cookbook repository or one of its subdirectories.")
ENV_FILE = REPO_ROOT / ".env.local"
load_dotenv(ENV_FILE)
assert os.getenv("OPENAI_API_KEY"), f"Add OPENAI_API_KEY to {ENV_FILE}."

EXAMPLE_DIR = REPO_ROOT / "examples/agents_api/build-observable-store-replenishment"
DATA_DIR = EXAMPLE_DIR / "data"
sys.path.insert(0, str(EXAMPLE_DIR))

import replenishment_agent

# Reload local code when this setup cell is rerun in an existing kernel.
importlib.reload(replenishment_agent)

from replenishment_agent import (
    AGENT_INSTRUCTIONS,
    FOLLOW_UP_INPUT,
    INITIAL_INPUT,
    MODEL,
    PLATFORM_LOGS_URL,
    SCENARIO_VERSION,
    TOOL_DEFINITIONS,
    ScenarioData,
    ToolCall,
    TurnResult,
    continue_after_storm,
    parse_decision,
    start_incident,
)

## 1. Inspect the synthetic dataset

The dataset is synthetic so the example is small, reproducible, and safe to share. [`generate_dataset.py`](build-observable-store-replenishment/generate_dataset.py) produces the committed files deterministically.

| File | Production system it represents |
| --- | --- |
| `inventory.json` | Shelf and back-room inventory system |
| `demand_forecast.json` | Forecasting service |
| `inbound_shipments.json` | Warehouse or transportation system |
| `weather_events.json` | External disruption feed |
| `nearby_store_inventory.json` | Store network inventory |
| `replenishment_policy.md` | Retailer's decision policy |

There is one incident: `store_101` is replenishing `water_24pk`.

In [ ]:
def read_json(filename):
    return json.loads((DATA_DIR / filename).read_text(encoding="utf-8"))


inventory = read_json("inventory.json")
forecast = read_json("demand_forecast.json")
shipment = read_json("inbound_shipments.json")[0]
storm = read_json("weather_events.json")[0]
nearby = read_json("nearby_store_inventory.json")[0]

display(
    Markdown(
        "\n".join(
            [
                "### Starting facts",
                "",
                "| Fact | Value |",
                "| --- | ---: |",
                f"| Shelf units | {inventory[0]['shelf_units']} |",
                f"| Back-room units | {inventory[0]['backroom_units']} |",
                f"| Shelf capacity | {inventory[0]['shelf_capacity']} |",
                f"| Baseline 24-hour demand | {forecast['baseline_units']} |",
                f"| Inbound shipment | {shipment['units']} units |",
                f"| Storm delay | {storm['shipment_delay_hours']} hours |",
                f"| Storm demand | {storm['revised_demand_units']} units |",
                f"| Nearby units available to transfer | {nearby['available_transfer_units']} |",
            ]
        )
    )
)

## 2. Keep operational data behind application tools

The Agents API does not receive database credentials or direct access to inventory systems. The application defines six function tools. When the agent requests one, Python reads the relevant synthetic record and returns a JSON result.

In this example, `scenario = ScenarioData(DATA_DIR)` is an **in-memory stand-in for the store's backend systems**. It loads the synthetic files and exposes ordinary Python methods such as `get_inventory_position(...)` and `get_demand_forecast(...)`. It is not an Agents API object and it does not make model calls.

Later, `scenario.call(action["name"], action["arguments"])` performs three straightforward steps:

1. Read the function name selected by the agent, such as `get_inventory_position`.
2. Route it to the matching Python method in `ScenarioData`.
3. Return that method's dictionary as `output`.

```text
agent requests get_inventory_position
        -> your Python function reads inventory data
        -> your application submits a tool_result
        -> the Agents API continues the turn
```

In a production application, the same Python method could query an inventory API or database. The Agents API interaction would remain the same.

In [ ]:
# Local stand-in for the retailer's inventory, forecast, and shipment services.
scenario = ScenarioData(DATA_DIR)

print("TOOLS AVAILABLE TO THE AGENT\n")
for tool in TOOL_DEFINITIONS:
    print(f"- {tool['name']}: {tool['description']}")

print("\nEXAMPLE RESULT RETURNED BY THE STORE APPLICATION\n")
print(json.dumps(scenario.get_inventory_position("store_101", "water_24pk"), indent=2))

## 3. Handle Agents API function calls

This is the center of the example. The event and payload names below come from the official [function tools](https://developers.openai.com/api/docs/guides/agents-api/tools/functions) and [session events](https://developers.openai.com/api/docs/guides/agents-api/sessions/events) documentation. The installed `openai` package also provides typed Python classes for these events.

The complete function is shown in the next cell. Read it in these stages:

| Code | What it does | Defined by |
| --- | --- | --- |
| `def collect_visible_turn(...)` | Wraps one streamed turn so this notebook can return a convenient result. | This example |
| `turn_id`, `text_parts`, `event_types`, `tool_calls` | Creates local Python variables for the completed turn, streamed text, debugging timeline, and displayed tool history. | This example |
| `for event in events` | Reads each typed event yielded by the OpenAI SDK stream. | Python SDK pattern |
| `event.type` | Identifies the kind of Agents API event. | Agents API |
| `agent.session.created` | Provides the new managed session and its ID. | Agents API |
| `agent.session.turn.in_progress` | Reports that work on the turn is underway. | Agents API |
| `agent.session.requires_action` | Pauses for input that the application must provide. | Agents API |
| `event.session.required_actions` | Lists the pending actions. The function-tools guide explicitly says to read calls here. | Agents API |
| `pending.to_dict()` | Converts the SDK's typed pending-action object into an ordinary Python dictionary. | OpenAI Python SDK |
| `scenario.call(...)` | Runs our application-owned inventory, forecast, shipment, weather, or policy function. | This example |
| `ToolCall(...)` | Saves a compact copy for the notebook's business-readable timeline. | This example |
| `sessions.events.create(...)` | Sends input back into the managed session. | OpenAI Python SDK |
| `agent.session.input.tool_result` | Marks that input as the result of a requested function. | Agents API |
| `turn_id` and `call_id` | Connect the result to the exact pending call. Both values must be copied from that action. | Agents API |
| `success=True` | Tells the API that the application function completed successfully. | Agents API |
| `json.dumps(output)` | Serializes our Python dictionary because a JSON tool result is submitted as a string. | Python plus Agents API requirement |
| `agent.session.turn.output_text.delta` | Carries a streamed piece of the agent's text. | Agents API |
| `agent.session.turn.output_text.done` | Provides the complete text; the fallback matters because text deltas may be absent. | Agents API |
| `agent.session.turn.completed` | Confirms successful completion. `subagent_id is None` selects the root turn. | Agents API |
| failure and cancellation events | Stop the application instead of treating a broken stream as success. | Agents API |
| `sessions.retrieve(...)` after an early stream close | Reads the durable session state. If it is paused at `requires_action`, the pending calls are still recoverable. | OpenAI Python SDK |
| `sessions.events.stream(...)` | Reconnects to the existing session before the application submits the recovered tool results, so continuation events are not missed. | OpenAI Python SDK |
| `parse_decision(...)` and `TurnResult(...)` | Validate the final JSON and package the fields used later in this notebook. | This example |

The most important boundary is therefore: **OpenAI defines the event names, pending-action fields, SDK method, and tool-result schema; this application defines `scenario.call`, `ToolCall`, `parse_decision`, and `TurnResult`.** The small reconnect branch matters because sessions are durable even when a particular HTTP stream closes: the application can retrieve the saved state and continue the same turn.

In [ ]:
def collect_visible_turn(client, events, scenario, session_id=None, state=None):
    turn_id = None
    if state is None:
        state = {}
    state.setdefault("text_parts", [])
    state.setdefault("event_types", [])
    state.setdefault("tool_calls", [])
    state.setdefault("handled_call_ids", set())
    state.setdefault("reported_progress", set())

    def submit_function_results(required_actions):
        submitted = 0
        for pending in required_actions:
            action = pending.to_dict()
            if (
                action["type"] != "function_call"
                or action["call_id"] in state["handled_call_ids"]
            ):
                continue

            state["handled_call_ids"].add(action["call_id"])
            print(f"\n[function requested] {action['name']}")
            print("Arguments:")
            print(json.dumps(action["arguments"], indent=2))

            # This is our application code, not code run by the model.
            output = scenario.call(action["name"], action["arguments"])
            print("Result returned by the store application:")
            print(json.dumps(output, indent=2))
            state["tool_calls"].append(
                ToolCall(action["name"], action["arguments"], output)
            )
            client.beta.agents.sessions.events.create(
                session_id,
                events=[
                    {
                        "type": "agent.session.input.tool_result",
                        "turn_id": action["turn_id"],
                        "call_id": action["call_id"],
                        "success": True,
                        "output": json.dumps(output),
                    }
                ],
            )
            print("[tool result submitted] The agent can now use this evidence.")
            submitted += 1
        return submitted

    for event in events:
        state["event_types"].append(event.type)

        if event.type == "agent.session.created":
            session_id = event.session.id
            state["session_id"] = session_id
            marker = f"session:{session_id}"
            if marker not in state["reported_progress"]:
                print(f"[event] session created: {session_id}")
                state["reported_progress"].add(marker)

        elif event.type == "agent.session.turn.in_progress":
            if "turn_started" not in state["reported_progress"]:
                print("[event] turn started")
                state["reported_progress"].add("turn_started")

        elif event.type == "agent.session.requires_action":
            session_id = session_id or event.session.id
            unhandled = [
                pending
                for pending in event.session.required_actions
                if pending.to_dict().get("call_id")
                not in state["handled_call_ids"]
            ]
            if unhandled:
                print(
                    f"[event] agent requires {len(unhandled)} "
                    "application function result(s)"
                )
                submit_function_results(unhandled)

        elif event.type == "agent.session.turn.output_text.delta":
            state["text_parts"].append(event.delta)

        elif (
            event.type == "agent.session.turn.output_text.done"
            and not state["text_parts"]
        ):
            state["text_parts"].append(event.text)

        elif event.type == "agent.session.turn.completed":
            if event.turn.subagent_id is None:
                turn_id = event.turn.id
                print(f"\n[event] turn completed: {turn_id}")
                break

        elif event.type in {
            "error",
            "agent.session.failed",
            "agent.session.environment.failed",
        }:
            raise RuntimeError(f"Agent failed: {event.type}")

        elif event.type in {
            "agent.session.turn.failed",
            "agent.session.turn.cancelled",
        }:
            if event.turn.subagent_id is None:
                raise RuntimeError(f"Agent failed: {event.type}")

    if session_id is None:
        raise RuntimeError("The stream ended before a session was created.")

    if turn_id is None:
        print("[stream] connection ended; retrieving the durable session state")
        saved_session = client.beta.agents.sessions.retrieve(session_id)
        if saved_session.status == "failed":
            raise RuntimeError(saved_session.error or "The session failed.")
        if saved_session.status != "requires_action":
            raise RuntimeError(
                f"The stream ended with session status {saved_session.status}."
            )

        print(f"[session] saved status: {saved_session.status}")
        # Subscribe first, then answer the persisted pending calls.
        with client.beta.agents.sessions.events.stream(session_id) as continuation:
            submitted = submit_function_results(saved_session.required_actions)
            if submitted == 0:
                raise RuntimeError("No unhandled function calls were found.")
            return collect_visible_turn(
                client, continuation, scenario, session_id, state
            )

    final_text = "".join(state["text_parts"])
    return TurnResult(
        session_id=session_id,
        turn_id=turn_id,
        final_text=final_text,
        decision=parse_decision(final_text),
        event_types=tuple(dict.fromkeys(state["event_types"])),
        tool_calls=tuple(state["tool_calls"]),
    )

### How `requires_action` and `required_actions` fit together

`agent.tools` is the catalog of tools available to the agent. By contrast, `event.session.required_actions` contains only the work the agent selected and is currently waiting for. It is a pending-work list, not a tool catalog.

For example, suppose an application gives the agent a `generate_invoice` function tool. After reading a request, the agent may select that tool. The Agents API then emits an event shaped like this:

```json
{
  "type": "agent.session.requires_action",
  "session": {
    "required_actions": [
      {
        "type": "function_call",
        "name": "generate_invoice",
        "arguments": {"order_id": "order_123"},
        "turn_id": "turn_abc",
        "call_id": "call_xyz"
      }
    ]
  }
}
```

At this point, the agent has **requested** the function but has not executed it. `for pending in event.session.required_actions` iterates over the pending request above. The application must call its own `generate_invoice(order_id="order_123")`, then submit the result with the same `turn_id` and `call_id`. The Agents API uses those IDs to resume the waiting turn with the correct result.

A single `requires_action` event can contain more than one pending call. The loop handles every action in that event. More `requires_action` events can also arrive later if the agent needs additional evidence.

### Example events in this replenishment turn

The identifiers below are illustrative, but the event and payload shapes are the real API contract.

| Incoming event | Handler effect | Visible output or submitted value |
| --- | --- | --- |
| `agent.session.created` | Saves `event.session.id` in `session_id`. | `Session created: sess_123` |
| `agent.session.turn.in_progress` | Reports that the turn is running. | `Turn started` |
| `agent.session.requires_action` with `get_inventory_position` | `scenario.call(...)` reads the inventory file. | `Tool requested: get_inventory_position`; output includes `shelf_units: 4`, `backroom_units: 20`, and `shelf_capacity: 24`. |
| submitted `tool_result` | `sessions.events.create(...)` sends the serialized inventory result. | The SDK call returns no response body; the waiting turn continues. |
| another pending `get_demand_forecast` call | `scenario.call(...)` reads the forecast. | `Tool requested: get_demand_forecast`; output includes `forecast_units: 18`. |
| `agent.session.turn.output_text.delta` | Appends `event.delta` to `text_parts`. | Successive pieces might be `{"decision":`, `"restock_from_backroom",`, and the remaining JSON. |
| `agent.session.turn.output_text.done` | Uses the complete `event.text` only when no deltas arrived. | Prevents an empty result when the stream omits deltas. |
| `agent.session.turn.completed` | Saves the root `event.turn.id`. | `Turn completed: turn_abc` |
| function return | Joins all text pieces, validates the JSON, and returns `TurnResult`. | `decision=restock_from_backroom`, `quantity=16`, plus the session ID, turn ID, events, and tool calls. |

`agent.session.turn.output_text.delta` is therefore not a separate model decision. It is one incremental text fragment, useful for displaying output as it is generated. The handler collects all fragments and joins them into the final response.

## 4. Turn 1: respond to the low-shelf alert

### The business scenario

You are the manager of **Lakeside Market**. A low-shelf alert fires for the store's 24-pack bottled water:

- The sales-floor shelf has **4 units** and can hold 24.
- The back room has **20 units**.
- Policy says a shelf below 8 units should be refilled to the **20-unit presentation target**.
- Baseline demand is **18 units over the next 24 hours**.
- No storm is active yet.

The question for the agent is: **what should the store do now?** The agent cannot move inventory. It can request facts through the tools and recommend an action; the store application and manager retain control.

There are two different meanings of **event** in this section:

- The **business event** is the low-shelf alert sent as `INITIAL_INPUT`.
- The **Agents API events** are progress messages in the response stream, such as `session.created`, `requires_action`, and `turn.completed`. They let the Python application observe and participate in the turn.

### What the next cell does

1. `sessions.create(...)` creates one managed Agents API session and starts its first turn.
2. `input=INITIAL_INPUT` sends the low-shelf alert shown by the cell.
3. `tools=TOOL_DEFINITIONS` tells the agent which store functions it may request.
4. `environment={"type": "none"}` means there is no hosted computer or shell; this example only uses application functions.
5. `collect_visible_turn(...)` handles the event stream, executes requested functions in Python, prints their returned JSON, and collects the final decision.

The session ID is saved in `first_turn.session_id` so a later storm can continue the same incident.

In [ ]:
def show_first_turn(result):
    evidence = "\n".join(
        f"- {item}" for item in result.decision.evidence_used
    )
    tools_used = " -> ".join(call.name for call in result.tool_calls)
    events_seen = " -> ".join(dict.fromkeys(result.event_types))
    display(
        Markdown(
            f"### What the agent decided\n\n"
            f"**Decision:** `{result.decision.decision}`  \n"
            f"**Quantity:** {result.decision.quantity} units  \n"
            f"**Manager approval required:** {result.decision.approval_required}  \n"
            f"**Summary:** {result.decision.summary}\n\n"
            f"**Evidence cited by the agent**\n{evidence}\n\n"
            f"**Application functions used**  \n`{tools_used}`\n\n"
            f"**Agents API events observed**  \n`{events_seen}`\n\n"
            f"**Session:** `{result.session_id}`  \n"
            f"**Turn:** `{result.turn_id}`"
        )
    )


client = OpenAI()
first_turn = None
turn_state = {}

print(f"SCENARIO VERSION: {SCENARIO_VERSION}")
print("BUSINESS EVENT SENT TO THE AGENT")
print(INITIAL_INPUT.strip())
print("\nLIVE AGENTS API ACTIVITY")

try:
    with client.beta.agents.sessions.create(
        agent={
            "model": MODEL,
            "instructions": AGENT_INSTRUCTIONS,
            "tools": TOOL_DEFINITIONS,
        },
        environment={"type": "none"},
        input=INITIAL_INPUT,
        stream=True,
    ) as events:
        first_turn = collect_visible_turn(
            client, events, scenario, state=turn_state
        )
except APIError as error:
    body = error.body if isinstance(error.body, dict) else {}
    is_rate_limit = (
        getattr(error, "status_code", None) == 429
        or body.get("code") == "rate_limit_exceeded"
        or "rate limit" in str(error).lower()
    )
    if not is_rate_limit:
        raise

    partial_session_id = turn_state.get("session_id")
    if partial_session_id:
        try:
            client.beta.agents.sessions.delete(partial_session_id)
        except APIError:
            pass
    display(
        Markdown(
            "### Live request paused by the project rate limit\n\n"
            "The session started, but the API rejected further work before "
            "the agent requested any store functions. The scenario code has "
            "not failed. Wait for the project's rate-limit window to reset, "
            "then rerun **only this Turn 1 cell**. Do not run the manager "
            "approval cell until a 16-unit recommendation appears."
        )
    )

if first_turn is not None:
    first_turn_scenario_version = SCENARIO_VERSION
    show_first_turn(first_turn)

### Manager approval: apply the shelf move

The intended first decision is `restock_from_backroom` for **16 units**: the shelf needs `20 - 4 = 16` units to reach its presentation target, and the back room has exactly 20 available.

The agent only recommends this action. The following cell represents the manager approving it and the store application updating inventory. It shows both records so the state change is visible before the storm arrives.

In [ ]:
if first_turn is None:
    raise RuntimeError(
        "Turn 1 has no result. Wait for the rate-limit window to reset, "
        "then rerun the Turn 1 cell before recording manager approval."
    )

if globals().get("first_turn_scenario_version") != SCENARIO_VERSION:
    raise RuntimeError(
        "Turn 1 is still stored from an older notebook run. Rerun the "
        "setup, tool-collector, and Turn 1 cells before approving it."
    )

expected_decision = "restock_from_backroom"
expected_quantity = 16

assert (
    first_turn.decision.decision == expected_decision
    and first_turn.decision.quantity == expected_quantity
), (
    "This teaching scenario expects a 16-unit back-room refill. "
    f"Received {first_turn.decision.decision!r} for "
    f"{first_turn.decision.quantity} units instead."
)

before_approval = scenario.get_inventory_position(
    "store_101", "water_24pk"
)
display_before_shelf = before_approval["shelf_units"]
display_before_backroom = before_approval["backroom_units"]
if display_before_shelf == 20 and display_before_backroom == 4:
    display_before_shelf, display_before_backroom = 4, 20

# Record the manager's approval once, even if this cell is run again.
if before_approval["shelf_units"] == 4:
    scenario.apply_approved_restock(expected_quantity)

after_approval = scenario.get_inventory_position(
    "store_101", "water_24pk"
)

display(
    Markdown(
        "### Manager approved the 16-unit shelf move\n\n"
        "| Inventory location | Before approval | After approval | Change |\n"
        "| --- | ---: | ---: | ---: |\n"
        f"| Sales-floor shelf | {display_before_shelf} | "
        f"{after_approval['shelf_units']} | +{expected_quantity} |\n"
        f"| Back room | {display_before_backroom} | "
        f"{after_approval['backroom_units']} | -{expected_quantity} |\n\n"
        "The store now has **20 units on the shelf** and **4 units in "
        "the back room**. The same `scenario` object carries this updated "
        "state into the storm turn."
    )
)

## 5. Turn 2: continue after the storm event

### The business scenario has changed

The first manager action is already complete: the shelf increased from 4 to **20 units**, and the back room decreased from 20 to **4 units**. The `get_inventory_position` result therefore begins this turn with `shelf_units: 20` and `backroom_units: 4`; it is showing the result of the approved shelf move, not skipping it.

Now a severe storm creates a new decision:

- Local inventory is **24 units total**: 20 on the shelf plus 4 in the back room.
- Expected demand rises to **50 units** over 24 hours.
- The inbound truck is delayed **48 hours**, outside that demand window.
- The projected shortfall is `50 - 24 = 26` units.
- Hilltop Market has **40 units** available to transfer.

The question for the same agent session is now: **should the manager request inventory from the nearby store?** The next cell first displays this changed state, then streams the agent's reassessment.

In [ ]:
scenario.activate_storm()

storm_inventory = scenario.get_inventory_position(
    "store_101", "water_24pk"
)
storm_forecast = scenario.get_demand_forecast(
    "store_101", "water_24pk"
)
storm_shipment = scenario.get_inbound_shipment(
    "store_101", "water_24pk"
)
storm_nearby = scenario.get_nearby_inventory(
    "store_101", "water_24pk"
)
local_units = (
    storm_inventory["shelf_units"]
    + storm_inventory["backroom_units"]
)
projected_shortfall = max(
    0, storm_forecast["forecast_units"] - local_units
)

display(
    Markdown(
        "### Store position when the storm arrives\n\n"
        "| Operational fact | Value |\n"
        "| --- | ---: |\n"
        f"| Shelf inventory after approved refill | {storm_inventory['shelf_units']} units |\n"
        f"| Back-room inventory after approved refill | {storm_inventory['backroom_units']} units |\n"
        f"| Total local inventory | {local_units} units |\n"
        f"| Revised 24-hour demand | {storm_forecast['forecast_units']} units |\n"
        f"| Inbound delay | {storm_shipment['delay_hours']} hours |\n"
        f"| Projected shortfall | {projected_shortfall} units |\n"
        f"| Nearby units available | {storm_nearby['available_transfer_units']} units |\n\n"
        "The shelf move is complete. The new problem is covering the "
        "26-unit storm shortfall before the delayed truck arrives."
    )
)

print("STORM EVENT SENT TO THE SAME AGENT SESSION")
print(FOLLOW_UP_INPUT.strip())
print("\nLIVE AGENTS API ACTIVITY")
with client.beta.agents.sessions.stream(
    first_turn.session_id,
    input=FOLLOW_UP_INPUT,
) as events:
    storm_turn = collect_visible_turn(
        client,
        events,
        scenario,
        session_id=first_turn.session_id,
    )

assert storm_turn.session_id == first_turn.session_id
storm_turn_scenario_version = SCENARIO_VERSION

storm_evidence = "\n".join(
    f"- {item}" for item in storm_turn.decision.evidence_used
)
display(
    Markdown(
        f"### What the agent recommends after the storm\n\n"
        f"**Decision:** `{storm_turn.decision.decision}`  \n"
        f"**Quantity:** {storm_turn.decision.quantity} units  \n"
        f"**Manager approval required:** {storm_turn.decision.approval_required}  \n"
        f"**Summary:** {storm_turn.decision.summary}\n\n"
        f"**Why 26 units?**  \n"
        f"Demand 50 - local inventory 24 = **26-unit shortfall**.\n\n"
        f"**Evidence cited by the agent**\n{storm_evidence}\n\n"
        "No inventory has moved yet. The agent has requested a transfer; "
        "the manager must approve it next."
    )
)

### Manager approval: authorize the nearby-store transfer

The agent cannot authorize an inter-store transfer. In the next cell, the manager approves the recommended 26 units and the application reserves them at Hilltop Market. Approval does **not** instantly add those units to the Lakeside shelf; it creates an approved dispatch while the physical inventory remains in transit.

In [ ]:
def record_transfer_approval(turn):
    if globals().get("storm_turn_scenario_version") != SCENARIO_VERSION:
        display(
            Markdown(
                "### Rerun the storm turn first\n\n"
                "This result came from an older scenario version. Rerun the "
                "setup and both agent turns before recording approval."
            )
        )
        return None

    if not (
        turn.decision.decision == "request_store_transfer"
        and turn.decision.quantity == 26
    ):
        display(
            Markdown(
                "### Manager approval not recorded\n\n"
                f"The live agent returned `{turn.decision.decision}` for "
                f"{turn.decision.quantity} units. This scenario expects a "
                "26-unit transfer, so the application stopped instead of "
                "silently changing inventory. Rerun the storm turn with the "
                "current scenario version."
            )
        )
        return None

    nearby_before = scenario.get_nearby_inventory(
        "store_101", "water_24pk"
    )
    nearby_before_units = nearby_before["available_transfer_units"]
    if scenario.approved_transfer is not None:
        nearby_before_units += scenario.approved_transfer["quantity"]
    local_before = scenario.get_inventory_position(
        "store_101", "water_24pk"
    )
    transfer = scenario.apply_approved_transfer(26)
    nearby_after = scenario.get_nearby_inventory(
        "store_101", "water_24pk"
    )
    local_after = scenario.get_inventory_position(
        "store_101", "water_24pk"
    )
    display(
        Markdown(
            "### Manager approved the 26-unit transfer\n\n"
            "| Operational state | Before approval | After approval |\n"
            "| --- | ---: | ---: |\n"
            f"| Transfer approved for dispatch | 0 units | {transfer['quantity']} units |\n"
            f"| Hilltop units still available | {nearby_before_units} | "
            f"{nearby_after['available_transfer_units']} |\n"
            f"| Lakeside shelf units | {local_before['shelf_units']} | "
            f"{local_after['shelf_units']} |\n"
            f"| Lakeside back-room units | {local_before['backroom_units']} | "
            f"{local_after['backroom_units']} |\n\n"
            "The transfer is **approved for dispatch**. Lakeside remains "
            "at 24 local units until it arrives; Hilltop has 14 units still available."
        )
    )
    return transfer


approved_transfer = record_transfer_approval(storm_turn)

## 6. See the business outcome and application trace

The business view answers **what changed**. The tool view answers **what evidence the agent requested**. The session and turn IDs connect both views to the full Agents API trace.

In [ ]:
trace_table = "\n".join(
    [
        "### Replenishment timeline",
        "",
        f"Session: `{first_turn.session_id}`",
        "",
        "| Business step | Owner | Turn | Outcome | Quantity |",
        "| --- | --- | --- | --- | ---: |",
        f"| Low shelf alert | Agent | `{first_turn.turn_id}` | `{first_turn.decision.decision}` | {first_turn.decision.quantity} |",
        "| Shelf move approved: shelf 4 -> 20, back room 20 -> 4 | Manager + application | - | Inventory updated | 16 |",
        f"| Storm delays truck | Agent | `{storm_turn.turn_id}` | `{storm_turn.decision.decision}` | {storm_turn.decision.quantity} |",
        "| Nearby transfer approved: Hilltop availability 40 -> 14 | Manager + application | - | Approved for dispatch | 26 |",
        "",
        f"[Open the Agents trace in Platform Logs]({PLATFORM_LOGS_URL}) and search for the session ID above.",
    ]
)
display(Markdown(trace_table))

### What to inspect in Platform Logs

Open **Logs → Agents**, search for the session ID, and expand both turns. The trace shows model spans, function calls, function arguments and results, timing, status, and recorded token usage. The event stream above is live progress; the Platform trace is the completed record and may appear shortly after the turn finishes.

## 7. Try the customer playground

Change the storm delay, expected demand, or nearby inventory, then select **Run scenario**. Each run creates a fresh Agents API session, displays both recommendations, and prints the session ID for tracing. This makes a real API call. After inspecting the traces, select **Delete demo sessions**.

In [ ]:
import time


delay_control = widgets.IntSlider(
    value=48, min=0, max=72, step=12, description="Delay hours"
)
demand_control = widgets.IntSlider(
    value=50, min=10, max=80, step=2, description="Demand"
)
nearby_control = widgets.IntSlider(
    value=40, min=0, max=100, step=5, description="Nearby units"
)
run_button = widgets.Button(description="Run scenario", button_style="primary")
cleanup_button = widgets.Button(description="Delete demo sessions")
playground_output = widgets.Output()
playground_session_ids = []
playground_run_state = {"running": False, "accept_after": 0.0}


def run_playground(_):
    now = time.monotonic()
    if playground_run_state["running"] or now < playground_run_state["accept_after"]:
        return
    playground_run_state["running"] = True
    run_button.disabled = True
    try:
        with playground_output:
            clear_output(wait=True)
            display(Markdown("**Running both agent turns...**"))
            custom = ScenarioData(
                DATA_DIR,
                storm_delay_hours=delay_control.value,
                storm_demand_units=demand_control.value,
                nearby_transfer_units=nearby_control.value,
            )
            with OpenAI() as demo_client:
                first = start_incident(
                    demo_client, custom, progress=lambda _message: None
                )
                playground_session_ids.append(first.session_id)
                revised = continue_after_storm(
                    demo_client,
                    custom,
                    first,
                    progress=lambda _message: None,
                )
            inventory = custom.get_inventory_position(
                "store_101", "water_24pk"
            )
            local_units = (
                inventory["shelf_units"] + inventory["backroom_units"]
            )
            shortfall = max(0, demand_control.value - local_units)
            clear_output(wait=True)
            display(
                Markdown(
                    "### Playground result\n\n"
                    "| Step | Business state or decision |\n"
                    "| --- | --- |\n"
                    f"| Inputs | Delay {delay_control.value}h; demand {demand_control.value}; nearby {nearby_control.value} |\n"
                    f"| Agent: low shelf | `{first.decision.decision}` for {first.decision.quantity} units |\n"
                    f"| Manager approval | Shelf 4 -> {inventory['shelf_units']}; back room 20 -> {inventory['backroom_units']} |\n"
                    f"| Storm position | {local_units} local units; {shortfall}-unit shortfall |\n"
                    f"| Agent: storm | `{revised.decision.decision}` for {revised.decision.quantity} units |\n"
                    f"| Manager action | {'Approval required' if revised.decision.approval_required else 'No approval required'} |\n\n"
                    f"Session: `{first.session_id}`\n\n"
                    f"[Inspect this session trace]({PLATFORM_LOGS_URL})"
                )
            )
    except APIError as error:
        with playground_output:
            clear_output(wait=True)
            display(
                Markdown(
                    "### The live run could not finish\n\n"
                    f"`{error}`\n\n"
                    "If this is a rate limit, wait for the project window "
                    "to reset and select **Run scenario** again."
                )
            )
    except (RuntimeError, ValueError) as error:
        with playground_output:
            clear_output(wait=True)
            display(Markdown(f"### Scenario stopped safely\n\n`{error}`"))
    finally:
        playground_run_state["running"] = False
        playground_run_state["accept_after"] = time.monotonic() + 2.0
        run_button.disabled = False


def delete_playground_sessions(_):
    cleanup_button.disabled = True
    try:
        deleted = 0
        remaining = []
        with OpenAI() as cleanup_client:
            for session_id in playground_session_ids:
                try:
                    cleanup_client.beta.agents.sessions.delete(session_id)
                    deleted += 1
                except APIError:
                    remaining.append(session_id)
        playground_session_ids[:] = remaining
        with playground_output:
            display(
                Markdown(
                    f"Deleted **{deleted}** playground session(s). "
                    f"**{len(remaining)}** still need cleanup."
                )
            )
    finally:
        cleanup_button.disabled = False


run_button.on_click(run_playground, remove=True)
cleanup_button.on_click(delete_playground_sessions, remove=True)
run_button.on_click(run_playground)
cleanup_button.on_click(delete_playground_sessions)
display(
    widgets.VBox(
        [
            delay_control,
            demand_control,
            nearby_control,
            widgets.HBox([run_button, cleanup_button]),
        ]
    )
)
display(playground_output)

## 8. Govern the same scenario in CI/CD

The included GitHub Actions template keeps production credentials away from pull requests:

```text
pull request -> mocked dataset, tool, and session tests
approved main run -> live two-turn canary -> recommendations + trace_summary.json
```

Copy [`store-replenishment-canary.yml`](build-observable-store-replenishment/github-actions/store-replenishment-canary.yml) into `.github/workflows/` in the backend repository. Add `OPENAI_API_KEY` to a protected GitHub environment named `agents-api-canary`.

In [ ]:
# The trace identifiers are displayed above, so this demo session can be removed.
client.beta.agents.sessions.delete(first_turn.session_id)
client.close()
print("Demo session removed.")

## 9. Share the manager experience

The companion browser demo uses the same synthetic data, tools, and decisions in a store-manager workspace. From `build-observable-store-replenishment`, run:

```bash
uv run demo/main.py --port 8010
```

Open `http://127.0.0.1:8010`, approve the initial shelf move, then adjust the truck delay, expected demand, and nearby inventory. **Guided** mode is a repeatable local walkthrough. **Live agent** mode uses the Agents API, preserves both turns in one session, and exposes trace identifiers for Platform Logs. The API key stays on the Python server. See [`demo/README.md`](build-observable-store-replenishment/demo/README.md) for the step-by-step customer story.

After the approved shelf refill, the shelf contains 20 cases and the back room contains 4. The storm alert becomes the next decision for the same incident:

![Store-manager demo after the approved shelf refill. The shelf contains 20 cases, the back room contains 4, and the agent reports a severe-weather alert.](../../images/agents-api-store-replenishment-shelf-approved.png)

After the approved nearby-store transfer, the truck has unloaded 26 cases, the shelf contains 20, the back room contains 10, and Hilltop has 14 cases still available:

![Store-manager demo after the approved storm transfer. The resolved incident shows 20 shelf cases, 10 back-room cases, and 14 cases remaining at the nearby store.](../../images/agents-api-store-replenishment-transfer-complete.png)

## Conclusion

The application owned the inventory, forecasts, shipment events, policy, and approvals. The Agents API owned the managed session, turns, event stream, and trace. Function tools connected those two sides without exposing operational systems directly to the model.

The result is one production-shaped workflow that remains easy to explain: create a session, answer a low-stock event, continue after a storm, and inspect exactly which tools and evidence produced the revised recommendation.